# Etapa 3: modelos preentrenados con Twitter en español

Entrena solo con HumAID traducido al español (`data/processed/humaid_es/`) y compara tres modelos preentrenados con tweets en español, recomendados por el profesor, contra XLM-R large de las etapas anteriores.

| Modelo | Hugging Face | Qué es |
|---|---|---|
| BERTuit | `AIDA-UPM/BERTuit-base` | RoBERTa entrenado desde cero con 230 millones de tweets en español (Huertas-Tato et al., 2022, arXiv:2204.03465) |
| RoBERTuito | `pysentimiento/robertuito-base-uncased` | RoBERTa entrenado desde cero con 500 millones de tweets en español (Pérez et al., 2022, arXiv:2111.09453) |
| BERT colombiano | `johnatanebonilla/bert-colombiano-v1` | BETO con preentrenamiento continuado en un corpus de Twitter de Colombia |

Contenido:

1. Verificación de que los tres modelos cargan y de cómo tokenizan un tweet.
2. Screening con la receta de la etapa 1 (mean pooling, LP-FT, LLRD) y pocas épocas; gana el mejor F1-macro en val.
3. Entrenamiento final del ganador.
4. Comparación con las etapas 1 y 2.

XLM-R large no se reentrena: se compara con los resultados ya guardados de las etapas 1 y 2. La comparación directa es Chile 2014 (y el set colombiano, con `scripts/evaluate_external.py`), que es idéntico en todas las etapas. XLM-R de la etapa 2 se entrenó con inglés y español, es decir, con el doble de datos; eso se tiene en cuenta al interpretar la comparación.

Todo se entrena con la semilla 42 y el modelo final se guarda en `models/etapa3/`.

## 1. Repositorio, dependencias y datos

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/JuanTrujilloM/final-project-NN-DL.git"
# Branch with this notebook; switch to "main" once Third_Stage is merged.
BRANCH = "Third_Stage"

# On Kaggle the repo is cloned; locally the notebook runs from the repo root.
if Path("/kaggle").exists():
    if not Path("final-project-NN-DL").exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL], check=True)
    os.chdir("final-project-NN-DL")
elif Path.cwd().name == "notebooks":
    os.chdir("..")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
if not Path("data/processed/humaid_en/train.csv").exists():
    subprocess.run([sys.executable, "scripts/prepare_data.py"], check=True)
# The repo code (preprocessing shared with the app) is imported from src/.
sys.path.insert(0, str(Path.cwd() / "src"))
print("Directorio de trabajo:", Path.cwd())

In [ ]:
import shutil

import pandas as pd

TRANSLATION_DIR = Path("data/raw/humaid_events_translated")
TRANSLATION_FILES = ["train.csv", "val.csv", "test.csv", "test_intra.csv"]


# Finds the folder with the four translated CSVs among the Kaggle inputs.
def find_translation(root=Path("/kaggle/input")):
    for path in sorted(root.rglob("test_intra.csv")):
        folder = path.parent
        if all((folder / name).exists() for name in TRANSLATION_FILES):
            if "text_es" in pd.read_csv(folder / "train.csv", nrows=1).columns:
                return folder
    return None


if not (TRANSLATION_DIR / "train.csv").exists() and Path("/kaggle/input").exists():
    source = find_translation()
    if source is not None:
        TRANSLATION_DIR.mkdir(parents=True, exist_ok=True)
        for name in TRANSLATION_FILES:
            shutil.copy(source / name, TRANSLATION_DIR / name)
        print("Traducción copiada desde", source)

if not (TRANSLATION_DIR / "train.csv").exists():
    raise SystemExit("No encuentro la traducción: agrega humaid_es.zip como Dataset de Kaggle o cópiala a "
                     f"{TRANSLATION_DIR}")
if not Path("data/processed/humaid_es/train.csv").exists():
    subprocess.run([sys.executable, "scripts/prepare_humaid_es.py"], check=True)

## 2. Librerías y configuración

In [ ]:
import gc
import json
import platform
import random
import re
import shutil
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, DataCollatorWithPadding, get_linear_schedule_with_warmup

from humaid.data.preprocess import adapt_text

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


def set_all_seeds(seed):
    os.environ["PL_GLOBAL_SEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_all_seeds(SEED)

In [ ]:
DATA_DIR = Path("data/processed")
ES_DIR = DATA_DIR / "humaid_es"
RESULTS_DIR = Path("results/etapa3")
SELECTION_DIR = RESULTS_DIR / "seleccion"
PREVIOUS_DIRS = {1: Path("results/etapa1"), 2: Path("results/etapa2")}
MODELS_DIR = Path("models") / "checkpoints" / "etapa3"
EXPORT_DIR = Path("models") / "etapa3"
TEXT_COL = "text_clean"

# None to train with all data; a small number (e.g. 500) for a quick test.
LIMIT = None
# Skips runs that already have a resumen.json (useful if the session drops).
SKIP_DONE = True
# What to run; each block can go in a different session.
RUN = {"verificar": True, "screening": True, "final": True}

LABEL_NAMES = [
    "caution_and_advice",
    "displaced_people_and_evacuations",
    "infrastructure_and_utility_damage",
    "injured_or_dead_people",
    "missing_or_found_people",
    "not_humanitarian",
    "other_relevant_information",
    "requests_or_urgent_needs",
    "rescue_volunteering_or_donation_effort",
    "sympathy_and_support",
]
NUM_CLASSES = len(LABEL_NAMES)
LABEL2ID = {label: i for i, label in enumerate(LABEL_NAMES)}
for path in [RESULTS_DIR, SELECTION_DIR, MODELS_DIR, EXPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

## 3. Datos

- **Train**: HumAID traducido al español, mismos desastres que en las etapas 1 y 2.
- **Val** (early stopping, screening y selección del mejor modelo): val traducido.
- **Evaluación**: `val`, `test` y `test_intra` en español; `chile_2014` (español nativo, nunca visto) y `test_en`, el test de la etapa 1 en inglés original. `test_en` mide cuánto pierde un modelo entrenado solo en español cuando le llega un tweet en inglés: son los mismos tweets que `test`, en su idioma original.

In [ ]:
def read_csv(path, lang):
    df = pd.read_csv(path, dtype={"tweet_id": str})
    df = df.dropna(subset=[TEXT_COL]).reset_index(drop=True)
    df["label_id"] = df["label"].map(LABEL2ID).astype(int)
    df["lang"] = lang
    return df


def limit_rows(df):
    if LIMIT is None:
        return df.reset_index(drop=True)
    return df.sample(n=min(LIMIT, len(df)), random_state=SEED).reset_index(drop=True)


train_df = limit_rows(read_csv(ES_DIR / "train.csv", "es"))
eval_dfs = {role: limit_rows(read_csv(ES_DIR / f"{role}.csv", "es")) for role in ["val", "test", "test_intra"]}
eval_dfs["chile_2014"] = limit_rows(read_csv(DATA_DIR / "chile2014_es" / "test.csv", "es"))
eval_dfs["test_en"] = limit_rows(read_csv(DATA_DIR / "humaid_en" / "test.csv", "en"))
val_df = eval_dfs["val"]

# The preparation scripts already check this; it is repeated here so the notebook never trains with a leak.
eval_ids = set().union(*(set(df["tweet_id"]) for df in eval_dfs.values()))
assert not set(train_df["tweet_id"]) & eval_ids, "Hay tweets de evaluación en train"

print(f"train: {len(train_df):,}")
for name, df in eval_dfs.items():
    print(f"  {name}: {len(df):,}")

### 3.1 Receta y candidatos

La receta es la de la etapa 1 (`results/etapa1/seleccion/config_transformer.json`): mean pooling, LP-FT, LLRD 0,95 y pesos de clase `sqrt`. El learning rate es 2e-5, el que usó el screening de la etapa 1 con los modelos base; el 1e-5 de XLM-R large es propio de un modelo large.

Cada modelo recibe el texto con los marcadores que vio en su preentrenamiento (`adapt_text` en `src/humaid/data/preprocess.py`, la misma función que usa la app):

| Modelo | `text_style` | `@USER` / `HTTPURL` pasan a |
|---|---|---|
| BERTuit | `bertuit` | `<usr>` / `<url>`, tokens especiales de su tokenizador |
| RoBERTuito | `robertuito` | `@usuario` / `url`, más emojis a texto, risas como `jaja` y letras repetidas a 3, como `pysentimiento` |
| BERT colombiano | `bert_colombiano` | igual que RoBERTuito: el modelo no documenta un formato, y BETO no tiene emojis en su vocabulario y parte `@USER` y `HTTPURL` en pedazos |

El HumAID en español solo existe ya limpio con `clean_tweet` (`text_raw` es el original en inglés), así que la adaptación parte de `text_clean`.

BERTuit solo publica pesos de TensorFlow y `transformers` 5 ya no los carga, así que `scripts/convert_bertuit.py` los pasa a PyTorch (lee el `.h5` con `h5py`, sin TensorFlow) y comprueba que el modelo convertido predice las palabras enmascaradas.

In [ ]:
BERTUIT_DIR = Path("models/checkpoints/hf/bertuit-base")
if not (BERTUIT_DIR / "model.safetensors").exists():
    subprocess.run([sys.executable, "scripts/convert_bertuit.py", "--out", str(BERTUIT_DIR)], check=True)

FT_BASE = {
    "max_len": 128,
    "dropout": 0.1,
    "batch_size": 32,
    "lp_lr": 0.001,
    "lp_epochs": 20,
    "lp_patience": 3,
    "lr": 2e-05,
    "llrd": 0.95,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "epochs": 10,
    "patience": 3,
    "clip_grad": 1.0,
    "class_weights": "sqrt",
    "amp": True,
}
# model_hf is the Hugging Face id, kept in the results; model_name is what gets loaded.
FT_CANDIDATES = {
    "bertuit": {"model_name": str(BERTUIT_DIR), "model_hf": "AIDA-UPM/BERTuit-base", "text_style": "bertuit"},
    "robertuito": {"model_name": "pysentimiento/robertuito-base-uncased", "text_style": "robertuito"},
    "bert_colombiano": {"model_name": "johnatanebonilla/bert-colombiano-v1", "text_style": "bert_colombiano"},
}
SCREEN_EPOCHS = 3


def model_texts(df, hp):
    return [adapt_text(text, hp.get("text_style")) for text in df[TEXT_COL]]

## 4. Métricas y reporte

In [ ]:
# F1 over the classes present in the true labels.
def compute_metrics(labels, preds):
    present = sorted(set(labels))
    return {
        "f1_macro": f1_score(labels, preds, labels=present, average="macro", zero_division=0),
        "f1_weighted": f1_score(labels, preds, labels=present, average="weighted", zero_division=0),
        "accuracy": accuracy_score(labels, preds),
    }


# Saves metrics, per-class report and confusion matrix for one set.
def full_report(labels, preds, name, model_name, out_dir):
    present = sorted(set(labels))
    names = [LABEL_NAMES[i] for i in present]
    metrics = compute_metrics(labels, preds)
    metrics["n"] = len(labels)
    metrics["por_clase"] = classification_report(
        labels, preds, labels=present, target_names=names, output_dict=True, zero_division=0
    )
    (out_dir / f"metricas_{name}.json").write_text(json.dumps(metrics, indent=2, ensure_ascii=False))

    print(f"\n{'=' * 60}\n  {model_name} | {name} — F1-macro: {metrics['f1_macro']:.4f} | "
          f"F1 ponderado: {metrics['f1_weighted']:.4f} | accuracy: {metrics['accuracy']:.4f}\n{'=' * 60}")
    print(classification_report(labels, preds, labels=present, target_names=names, digits=4, zero_division=0))

    cm = confusion_matrix(labels, preds, labels=present)
    cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    fig, axes = plt.subplots(1, 2, figsize=(22, 9))
    for ax, data, fmt, title in zip(
        axes, [cm, cm_norm], ["d", ".2f"], ["Conteos absolutos", "Proporción por clase (recall)"]
    ):
        sns.heatmap(data, annot=True, fmt=fmt, cmap="Blues", xticklabels=names, yticklabels=names, ax=ax)
        ax.set_xlabel("Predicción")
        ax.set_ylabel("Real")
        ax.set_title(f"{model_name} | {name}\n{title}")
    plt.tight_layout()
    fig.savefig(out_dir / f"confusion_{name}.png", dpi=120)
    plt.show()
    plt.close(fig)
    return metrics

## 5. Callbacks

In [ ]:
# Saves the best model according to a monitored metric.
class ModelCheckpoint:
    def __init__(self, filepath, monitor="val_f1_macro", mode="max", verbose=True):
        self.filepath = filepath
        self.monitor = monitor
        self.verbose = verbose
        self.best = float("-inf") if mode == "max" else float("inf")
        self.mode = mode

    def step(self, metrics, model):
        value = metrics[self.monitor]
        improved = value > self.best if self.mode == "max" else value < self.best
        if improved:
            self.best = value
            torch.save(model.state_dict(), self.filepath)
            if self.verbose:
                print(f"  ✓ Mejor modelo guardado ({self.monitor}: {value:.4f})")
        return improved


# Stops training when the metric stops improving.
class EarlyStopping:
    def __init__(self, monitor="val_f1_macro", patience=5, min_delta=1e-4, mode="max"):
        self.monitor = monitor
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best = float("inf") if mode == "min" else float("-inf")
        self.counter = 0
        self.should_stop = False

    def step(self, metrics, model=None):
        value = metrics[self.monitor]
        improved = (value < self.best - self.min_delta if self.mode == "min"
                    else value > self.best + self.min_delta)
        if improved:
            self.best = value
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
                print(f"  ⏹ EarlyStopping activado (sin mejora en {self.patience} épocas)")
        return improved

## 6. Utilidades de entrenamiento

In [ ]:
# Class weights from train: balanced, sqrt (softer) or none.
def class_weights(label_ids, mode="balanced"):
    if mode == "none":
        return None
    counts = np.bincount(np.asarray(label_ids), minlength=NUM_CLASSES).astype(float)
    weights = len(label_ids) / (NUM_CLASSES * np.maximum(counts, 1))
    if mode == "sqrt":
        weights = np.sqrt(weights)
    return torch.tensor(weights, dtype=torch.float).to(DEVICE)


# One epoch; trains if an optimizer is given, otherwise only evaluates.
def run_epoch(model, loader, criterion, forward_fn, optimizer=None, scheduler=None, scaler=None, clip_grad=1.0):
    training = optimizer is not None
    model.train() if training else model.eval()
    use_amp = scaler is not None
    total_loss, n = 0.0, 0
    all_preds, all_labels = [], []

    with torch.set_grad_enabled(training):
        for batch in loader:
            with torch.autocast(device_type=DEVICE.type, enabled=use_amp):
                logits, labels = forward_fn(model, batch)
                loss = criterion(logits.float(), labels)

            if training:
                optimizer.zero_grad()
                if use_amp:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_grad)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_grad)
                    optimizer.step()
                if scheduler is not None:
                    scheduler.step()

            total_loss += loss.item() * labels.size(0)
            n += labels.size(0)
            all_preds.extend(logits.argmax(dim=1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    metrics = compute_metrics(all_labels, all_preds)
    return total_loss / n, metrics["accuracy"], metrics["f1_macro"]


@torch.no_grad()
def evaluate_clf(model, loader, forward_fn):
    model.eval()
    all_preds, all_labels = [], []
    for batch in loader:
        with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
            logits, labels = forward_fn(model, batch)
        all_preds.extend(logits.argmax(dim=1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
    return all_preds, all_labels


def new_history():
    return {"epoch": [], "phase": [], "lr": [], "train_loss": [], "val_loss": [],
            "train_acc": [], "val_acc": [], "train_f1_macro": [], "val_f1_macro": []}


def log_epoch(history, epoch, phase, lr, train, val):
    for key, value in [("epoch", epoch), ("phase", phase), ("lr", lr),
                       ("train_loss", train[0]), ("train_acc", train[1]), ("train_f1_macro", train[2]),
                       ("val_loss", val[0]), ("val_acc", val[1]), ("val_f1_macro", val[2])]:
        history[key].append(value)
    print(f"Época {epoch:02d} [{phase}] — "
          f"train loss: {train[0]:.4f}, F1-macro: {train[2]:.4f} | "
          f"val loss: {val[0]:.4f}, F1-macro: {val[2]:.4f}, acc: {val[1]:.4f} | lr: {lr:.2e}")
    return {"val_loss": val[0], "val_acc": val[1], "val_f1_macro": val[2]}


def plot_history(history, model_name, out_dir):
    pd.DataFrame(history).to_csv(out_dir / "history.csv", index=False)
    epochs = history["epoch"]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(epochs, history["train_loss"], label="train")
    axes[0].plot(epochs, history["val_loss"], label="val", linestyle="--")
    axes[1].plot(epochs, history["train_f1_macro"], label="train")
    axes[1].plot(epochs, history["val_f1_macro"], label="val", linestyle="--")
    axes[0].set_title(f"{model_name} — Loss")
    axes[1].set_title(f"{model_name} — F1-macro")
    for ax in axes:
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        ax.set_xlabel("Época")
        ax.legend()
        ax.grid(alpha=0.3)
    plt.tight_layout()
    fig.savefig(out_dir / "curvas.png", dpi=120)
    plt.show()
    plt.close(fig)


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total": total, "entrenables": trainable}


def hardware_info(start_time):
    info = {
        "tiempo_entrenamiento_min": round((time.time() - start_time) / 60, 2),
        "plataforma": platform.platform(),
        "torch": torch.__version__,
        "cpu_nucleos": os.cpu_count(),
        "gpu": None,
    }
    if DEVICE.type == "cuda":
        props = torch.cuda.get_device_properties(0)
        info["gpu"] = props.name
        info["gpu_memoria_gb"] = round(props.total_memory / 1e9, 1)
        info["gpu_memoria_max_usada_gb"] = round(torch.cuda.max_memory_allocated() / 1e9, 2)
    return info


# Evaluates the best model on every eval set; predictions are saved without tweet text.
def final_evaluation(model, make_loader, forward_fn, model_name, out_dir):
    results = {}
    for name, df in eval_dfs.items():
        preds, labels = evaluate_clf(model, make_loader(df), forward_fn)
        metrics = full_report(labels, preds, name, model_name, out_dir)
        results[name] = {key: metrics[key] for key in ("f1_macro", "f1_weighted", "accuracy", "n")}
        pd.DataFrame({
            "tweet_id": df["tweet_id"].astype(str),
            "event": df["event"],
            "lang": df["lang"],
            "label": [LABEL_NAMES[i] for i in labels],
            "pred": [LABEL_NAMES[i] for i in preds],
        }).to_csv(out_dir / f"predicciones_{name}.csv", index=False)
    return results


def save_summary(out_dir, **summary):
    (out_dir / "resumen.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False, default=str))


def load_summary(out_dir):
    path = out_dir / "resumen.json"
    return json.loads(path.read_text()) if SKIP_DONE and path.exists() else None


def free_memory():
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

## 7. Modelos

### 7.1 Verificación

Antes de entrenar: cada modelo carga, el tweet adaptado se tokeniza sin `<unk>` en los marcadores, y la longitud máxima cabe en sus posiciones (RoBERTuito admite 128 tokens). `unk %` es la proporción de tokens desconocidos en 2.000 tweets de train; un valor alto indica que el tokenizador no cubre bien el texto.

In [ ]:
class PretrainedTweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.encodings = tokenizer(list(texts), truncation=True, max_length=max_len)
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: self.encodings[key][idx] for key in ("input_ids", "attention_mask")}
        item["labels"] = self.labels[idx]
        return item


class PretrainedClassifier(nn.Module):
    def __init__(self, model_name, num_classes, dropout=0.1):
        super().__init__()
        # Always fp32 weights (some checkpoints load as fp16 by default); autocast handles mixed precision.
        self.encoder = AutoModel.from_pretrained(model_name, dtype=torch.float32)
        self.drop = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, num_classes)

    # Mean of the token vectors, ignoring padding.
    def pool(self, input_ids, attention_mask):
        hidden = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1).to(hidden.dtype)
        return (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

    def forward(self, input_ids, attention_mask):
        return self.classifier(self.drop(self.pool(input_ids, attention_mask)))

    def freeze_encoder(self):
        for p in self.encoder.parameters():
            p.requires_grad = False

    def unfreeze_encoder(self):
        for p in self.encoder.parameters():
            p.requires_grad = True


def forward_transformer(model, batch):
    logits = model(batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE))
    return logits, batch["labels"].to(DEVICE)

In [ ]:
def check_candidate(key, hp, sample):
    tokenizer = AutoTokenizer.from_pretrained(hp["model_name"])
    model = PretrainedClassifier(hp["model_name"], NUM_CLASSES, dropout=hp["dropout"])
    config = model.encoder.config
    # RoBERTa reserves pad_token_id + 1 positions before the first token.
    offset = config.pad_token_id + 1 if config.model_type == "roberta" else 0
    max_tokens = config.max_position_embeddings - offset
    assert hp["max_len"] <= max_tokens, f"{key}: max_len {hp['max_len']} > {max_tokens}"

    params = count_parameters(model)["total"] / 1e6
    example = adapt_text("@USER Necesitamos agua y comida en el albergue 🙏 HTTPURL", hp.get("text_style"))
    print(f"\n{key} ({config.model_type}, {params:.0f} M parámetros, "
          f"máximo {max_tokens} tokens)")
    print("  texto: ", example)
    print("  tokens:", tokenizer.convert_ids_to_tokens(tokenizer(example)["input_ids"]))

    ids = tokenizer(model_texts(sample, hp), truncation=True, max_length=hp["max_len"])["input_ids"]
    total = sum(len(seq) for seq in ids)
    unk = sum(seq.count(tokenizer.unk_token_id) for seq in ids)
    truncated = sum(len(seq) == hp["max_len"] for seq in ids)
    del model
    free_memory()
    return {"candidato": key, "tipo": config.model_type, "parametros_M": round(params),
            "max_tokens": max_tokens, "tokens_por_tweet": round(total / len(ids), 1),
            "unk %": round(100 * unk / total, 3), "truncados %": round(100 * truncated / len(ids), 2)}


if RUN["verificar"]:
    sample = train_df.sample(n=min(2000, len(train_df)), random_state=SEED)
    checks = pd.DataFrame([check_candidate(key, {**FT_BASE, **params}, sample)
                           for key, params in FT_CANDIDATES.items()])
    checks.to_csv(SELECTION_DIR / "verificacion.csv", index=False)
    display(checks)

In [ ]:
# Layer-wise lr decay: the top layer gets lr, each layer below gets lr * decay; no weight decay on biases or LayerNorm.
def llrd_param_groups(model, lr, decay, weight_decay):
    num_layers = model.encoder.config.num_hidden_layers
    groups = {}
    for name, param in model.named_parameters():
        if name.startswith("encoder.embeddings."):
            depth = 0
        elif match := re.match(r"encoder\.encoder\.layer\.(\d+)\.", name):
            depth = int(match.group(1)) + 1
        else:
            depth = num_layers + 1
        no_decay = param.ndim == 1
        group = groups.setdefault((depth, no_decay), {
            "params": [], "lr": lr * decay ** (num_layers + 1 - depth),
            "weight_decay": 0.0 if no_decay else weight_decay,
        })
        group["params"].append(param)
    return list(groups.values())


# Pooled vectors of the frozen encoder, computed once for the linear probe.
@torch.no_grad()
def extract_features(model, loader):
    model.eval()
    feats, labels = [], []
    for batch in loader:
        with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
            pooled = model.pool(batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE))
        feats.append(pooled.float())
        labels.append(batch["labels"].to(DEVICE))
    return torch.cat(feats), torch.cat(labels)


# Phase 1 (linear probe): trains only the classifier on the cached vectors until val F1 stops improving.
def train_head(model, train_feats, val_feats, criterion, hp, history):
    head_params = list(model.classifier.parameters())
    optimizer = torch.optim.Adam(head_params, lr=hp["lp_lr"])
    early_stop = EarlyStopping(patience=hp["lp_patience"])
    best_f1, best_state = float("-inf"), None
    (x_train, y_train), (x_val, y_val) = train_feats, val_feats
    for epoch in range(1, hp["lp_epochs"] + 1):
        model.drop.train()
        order = torch.randperm(len(y_train), device=DEVICE)
        total, preds = 0.0, torch.empty_like(y_train)
        for start in range(0, len(order), 256):
            idx = order[start:start + 256]
            logits = model.classifier(model.drop(x_train[idx]))
            loss = criterion(logits, y_train[idx])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(idx)
            preds[idx] = logits.argmax(dim=1)
        model.drop.eval()
        with torch.no_grad():
            val_logits = model.classifier(x_val)
            val_loss = criterion(val_logits, y_val).item()
        train_m = compute_metrics(y_train.tolist(), preds.tolist())
        val_m = compute_metrics(y_val.tolist(), val_logits.argmax(dim=1).tolist())
        metrics = log_epoch(history, epoch, "lp", hp["lp_lr"],
                            (total / len(y_train), train_m["accuracy"], train_m["f1_macro"]),
                            (val_loss, val_m["accuracy"], val_m["f1_macro"]))
        if metrics["val_f1_macro"] > best_f1:
            best_f1 = metrics["val_f1_macro"]
            best_state = {k: v.detach().clone() for k, v in model.classifier.state_dict().items()}
        early_stop.step(metrics)
        if early_stop.should_stop:
            break
    model.classifier.load_state_dict(best_state)
    return best_f1

In [ ]:
# Everything needed to rebuild the classifier outside the notebook: weights, encoder configuration and tokenizer.
def save_transformer(model, tokenizer, hp, model_dir, val_f1, seed):
    model_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), model_dir / "model.pt")
    model.encoder.config.save_pretrained(model_dir / "encoder")
    tokenizer.save_pretrained(model_dir / "tokenizer")
    config = {"tipo": "transformer", "semilla": seed, "mejor_val_f1_macro": val_f1, "labels": LABEL_NAMES,
              "hiperparametros": hp}
    (model_dir / "config.json").write_text(json.dumps(config, indent=2, ensure_ascii=False))
    print(f"Modelo guardado en {model_dir}")


# LP-FT: linear probe, then the whole model with LLRD; the best model by val F1-macro is kept.
def finetune(hp, run_name, out_dir, seed, full_eval, export_dir=None):
    set_all_seeds(seed)
    out_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = MODELS_DIR / f"{run_name}.pt"
    use_amp = hp["amp"] and DEVICE.type == "cuda"

    tokenizer = AutoTokenizer.from_pretrained(hp["model_name"], **hp.get("tokenizer_kwargs", {}))
    collator = DataCollatorWithPadding(tokenizer=tokenizer)
    num_workers = 2 if DEVICE.type == "cuda" else 0

    def make_loader(df, shuffle=False):
        dataset = PretrainedTweetDataset(model_texts(df, hp), df["label_id"], tokenizer, hp["max_len"])
        return DataLoader(dataset, batch_size=hp["batch_size"], shuffle=shuffle,
                          collate_fn=collator, num_workers=num_workers)

    train_loader, val_loader = make_loader(train_df, shuffle=True), make_loader(val_df)
    model = PretrainedClassifier(hp["model_name"], NUM_CLASSES, dropout=hp["dropout"]).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights(train_df["label_id"], hp["class_weights"]))
    checkpoint = ModelCheckpoint(str(ckpt_path))
    history = new_history()
    start_time = time.time()
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    print(f"\n[{run_name}] Fase 1 — encoder congelado (linear probe)")
    model.freeze_encoder()
    train_feats = extract_features(model, make_loader(train_df))
    val_feats = extract_features(model, val_loader)
    lp_f1 = train_head(model, train_feats, val_feats, criterion, hp, history)
    del train_feats, val_feats
    checkpoint.step({"val_f1_macro": lp_f1}, model)
    lp_epochs = len(history["epoch"])

    model.unfreeze_encoder()
    print(f"\n[{run_name}] Fase 2 — fine-tuning completo, {count_parameters(model)['entrenables']:,} parámetros")
    optimizer = torch.optim.AdamW(llrd_param_groups(model, hp["lr"], hp["llrd"], hp["weight_decay"]))
    total_steps = len(train_loader) * hp["epochs"]
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps * hp["warmup_ratio"]), num_training_steps=total_steps
    )
    scaler = torch.amp.GradScaler("cuda") if use_amp else None
    early_stop = EarlyStopping(patience=hp["patience"])
    for epoch in range(1, hp["epochs"] + 1):
        train_m = run_epoch(model, train_loader, criterion, forward_transformer, optimizer,
                            scheduler=scheduler, scaler=scaler, clip_grad=hp["clip_grad"])
        val_m = run_epoch(model, val_loader, criterion, forward_transformer, scaler=scaler)
        metrics = log_epoch(history, lp_epochs + epoch, "ft", optimizer.param_groups[-1]["lr"], train_m, val_m)
        checkpoint.step(metrics, model)
        early_stop.step(metrics)
        if early_stop.should_stop:
            break

    resources = hardware_info(start_time)
    plot_history(history, run_name, out_dir)
    summary = {"nombre": run_name, "modelo": hp.get("model_hf", hp["model_name"]), "mejor_val_f1_macro": checkpoint.best,
               "val_f1_macro_lp": lp_f1, "epocas_lp": lp_epochs, "epocas_ft": len(history["epoch"]) - lp_epochs,
               "epocas_entrenadas": len(history["epoch"]), "parametros": count_parameters(model),
               "recursos": resources, "hiperparametros": hp}
    if full_eval or export_dir is not None:
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    if full_eval:
        summary["resultados"] = final_evaluation(model, make_loader, forward_transformer, run_name, out_dir)
    save_summary(out_dir, **summary)
    if export_dir is not None:
        save_transformer(model, tokenizer, hp, export_dir, checkpoint.best, seed)
    print(f"Recursos: {resources}")

    del model, optimizer
    ckpt_path.unlink(missing_ok=True)
    free_memory()
    return summary

### 7.2 Screening

Cada candidato se entrena con la misma receta y solo `SCREEN_EPOCHS` épocas de fine-tuning, como en la etapa 1; gana el de mayor F1-macro en val. Test y Chile no se miran hasta el modelo final.

In [ ]:
def select_finetune(hp, run_name):
    out_dir = SELECTION_DIR / run_name
    if summary := load_summary(out_dir):
        print(f"{run_name}: ya tiene resultados, val F1 {summary['mejor_val_f1_macro']:.4f}")
        return summary
    return finetune(hp, run_name, out_dir, SEED, full_eval=False)


if RUN["screening"]:
    rows = []
    for key, params in FT_CANDIDATES.items():
        summary = select_finetune({**FT_BASE, **params, "epochs": SCREEN_EPOCHS}, f"ft_{key}")
        rows.append({"candidato": key, "modelo": summary["modelo"],
                     "val_f1_macro": round(summary["mejor_val_f1_macro"], 4),
                     "val_f1_macro_lp": round(summary["val_f1_macro_lp"], 4),
                     "minutos": summary["recursos"]["tiempo_entrenamiento_min"]})
    screening = pd.DataFrame(rows).sort_values("val_f1_macro", ascending=False)
    screening.to_csv(SELECTION_DIR / "transformers.csv", index=False)
    display(screening)

### 7.3 Modelo final

El ganador del screening con la receta completa (hasta 10 épocas de fine-tuning con early stopping). Queda en `models/etapa3/<modelo>_es/` con su `text_style`, para que la app adapte el texto igual que aquí.

In [ ]:
# Final run: trains with SEED and evaluates on every set; with export, the model is saved in EXPORT_DIR.
def final_finetune(hp, family, export=True):
    out_dir, model_dir = RESULTS_DIR / family, EXPORT_DIR / family
    if load_summary(out_dir) and (not export or (model_dir / "model.pt").exists()):
        print(f"{family}: ya tiene resultados, se omite")
        return
    summary = finetune(hp, family, out_dir, SEED, full_eval=True, export_dir=model_dir if export else None)
    save_summary(out_dir, **summary, familia=family, semilla=SEED, tipo="fine_tuning")



if RUN["final"]:
    winner = pd.read_csv(SELECTION_DIR / "transformers.csv").iloc[0]["candidato"]
    print("Ganador del screening:", winner)
    final_finetune({**FT_BASE, **FT_CANDIDATES[winner]}, f"{winner}_es")

## 8. Comparación con las etapas 1 y 2

`comparacion.csv` tiene el modelo final de la etapa 3 y `comparacion_etapas.csv` junta los modelos finales de las tres etapas. Qué se puede comparar:

- **`chile_2014`**: idéntico en las tres etapas, es la comparación directa.
- **`test`**: en la etapa 1 es inglés, en la 2 inglés y español juntos, en la 3 solo español; no es comparable entre etapas.
- **`test_en`** de la etapa 3 frente a `test` de la etapa 1: los mismos tweets en inglés, para ver cuánto se pierde en inglés al entrenar solo con español.

In [ ]:
def runs_table(results_dir, stage):
    rows = []
    for path in sorted(results_dir.glob("*/resumen.json")):
        summary = json.loads(path.read_text())
        if "resultados" not in summary:
            continue
        row = {"etapa": stage, "modelo": summary["familia"], "tipo": summary["tipo"],
               "épocas": summary["epocas_entrenadas"], "minutos": summary["recursos"]["tiempo_entrenamiento_min"],
               "gpu": summary["recursos"]["gpu"]}
        row.update({f"F1 {name}": round(r["f1_macro"], 4) for name, r in summary["resultados"].items()})
        row.update({f"F1 ponderado {name}": round(r["f1_weighted"], 4) for name, r in summary["resultados"].items()})
        rows.append(row)
    return pd.DataFrame(rows)



stage3 = runs_table(RESULTS_DIR, 3)
stage3.drop(columns="etapa").to_csv(RESULTS_DIR / "comparacion.csv", index=False)

previous = [runs_table(path, stage) for stage, path in PREVIOUS_DIRS.items()]
all_stages = pd.concat(previous + [stage3], ignore_index=True)
all_stages.to_csv(RESULTS_DIR / "comparacion_etapas.csv", index=False)
all_stages

In [ ]:
# Spanish of the real world: Chile 2014 (and colombia once scripts/evaluate_external.py has been run).
focus = [c for c in ["F1 chile_2014", "F1 ponderado chile_2014", "F1 colombia", "F1 ponderado colombia"]
         if c in all_stages]
all_stages[all_stages["tipo"] == "fine_tuning"][["etapa", "modelo", "minutos"] + focus]

In [ ]:
# English: test_en (stage 3) holds the same tweets as test of stage 1.
english = all_stages[all_stages["etapa"] == 1][["etapa", "modelo", "F1 test"]].rename(columns={"F1 test": "F1 inglés"})
english = pd.concat([english, stage3[["etapa", "modelo", "F1 test_en"]].rename(columns={"F1 test_en": "F1 inglés"})])
english

In [ ]:
if Path("/kaggle").exists():
    shutil.make_archive("/kaggle/working/resultados_etapa3", "zip", RESULTS_DIR)
    shutil.make_archive("/kaggle/working/modelos_etapa3", "zip", EXPORT_DIR)
    print("Resultados en /kaggle/working/resultados_etapa3.zip y modelos en /kaggle/working/modelos_etapa3.zip")